# Exploração dos Dados — SMS Spam Collection

Este notebook explora a base de dados utilizada no estudo comparativo de mecanismos de atenção em redes LSTM para detecção de spam.

**Objectivos:**
- Inspecionar a distribuição de classes
- Analisar comprimentos de mensagens
- Verificar duplicatas e integridade
- Explorar o vocabulário
- Visualizar exemplos representativos

**Nota:** Este notebook é exclusivamente exploratório. Não executa treinamento de modelos nem utiliza dados de validação ou teste para qualquer ajuste.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

REPO_ROOT = Path('..').resolve()
sys.path.insert(0, str(REPO_ROOT))

from src.data import (
    check_invalid_labels,
    check_missing_values,
    handle_duplicates,
    load_raw_data,
)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

RAW_PATH = REPO_ROOT / 'data' / 'raw' / 'SMSSpamCollection'
print(f'Dataset path: {RAW_PATH}')
print(f'File exists: {RAW_PATH.exists()}')

## 1. Carregamento e Visão Geral

In [ ]:
df = load_raw_data(RAW_PATH)
print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head(10)

In [ ]:
print('=== Tipos de dados ===')
print(df.dtypes)
print()
print('=== Valores ausentes ===')
missing = check_missing_values(df)
print(missing)
print()
print('=== Rótulos inválidos ===')
invalid = check_invalid_labels(df)
print(f'Linhas com rótulos inválidos: {len(invalid)}')

## 2. Distribuição de Classes

In [ ]:
class_counts = df['label'].value_counts()
class_pct = df['label'].value_counts(normalize=True) * 100

print('Contagem por classe:')
for label in ['ham', 'spam']:
    print(f'  {label:4s}: {class_counts[label]:5d}  ({class_pct[label]:.1f}%)')

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].bar(['ham', 'spam'], [class_counts['ham'], class_counts['spam']],
            color=['steelblue', 'tomato'], edgecolor='white')
axes[0].set_title('Contagem por classe')
axes[0].set_ylabel('Número de mensagens')
for i, v in enumerate([class_counts['ham'], class_counts['spam']]):
    axes[0].text(i, v + 20, str(v), ha='center', fontweight='bold')

axes[1].pie(
    [class_counts['ham'], class_counts['spam']],
    labels=['ham', 'spam'],
    colors=['steelblue', 'tomato'],
    autopct='%1.1f%%',
    startangle=90,
)
axes[1].set_title('Proporção de classes')

fig.suptitle('Distribuição de classes — SMS Spam Collection', fontsize=13)
fig.tight_layout()
plt.show()

## 3. Comprimento das Mensagens

In [ ]:
df['char_length'] = df['text'].str.len()
df['word_count']  = df['text'].str.split().str.len()

print('Comprimento em caracteres e palavras por classe:')
print(df.groupby('label')[['char_length', 'word_count']].agg(['mean', 'median', 'max', 'std']).round(1))

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for col, row_idx, title, xlabel in [
    ('char_length', 0, 'Comprimento em caracteres', 'Caracteres'),
    ('word_count',  1, 'Contagem de palavras', 'Palavras'),
]:
    for j, (label, color) in enumerate([('ham', 'steelblue'), ('spam', 'tomato')]):
        axes[row_idx][j].hist(
            df[df['label'] == label][col],
            bins=40, color=color, alpha=0.8, edgecolor='white'
        )
        axes[row_idx][j].set_title(f'{title} — {label}')
        axes[row_idx][j].set_xlabel(xlabel)
        axes[row_idx][j].set_ylabel('Frequência')
        median_val = df[df['label'] == label][col].median()
        axes[row_idx][j].axvline(median_val, color='black', linestyle='--',
                                  label=f'Mediana: {median_val:.0f}')
        axes[row_idx][j].legend()

fig.suptitle('Distribuição do comprimento das mensagens', fontsize=13)
fig.tight_layout()
plt.show()

In [ ]:
max_length = 100
pct_within_limit = (df['word_count'] <= max_length).mean() * 100
print(f'Mensagens com <= {max_length} palavras: {pct_within_limit:.1f}%')
print(f'Comprimento máximo: {df["word_count"].max()} palavras')
print(f'Percentil 95: {df["word_count"].quantile(0.95):.0f} palavras')
print(f'Percentil 99: {df["word_count"].quantile(0.99):.0f} palavras')

## 4. Duplicatas

In [ ]:
df_clean, dup_report = handle_duplicates(df)

print('=== Relatório de deduplicação ===')
print(f'  Total original               : {dup_report["n_original"]}')
print(f'  Textos conflitantes          : {dup_report["n_conflicting_texts"]} textos, {dup_report["n_rows_removed_conflicting"]} linhas')
print(f'  Duplicatas exactas removidas : {dup_report["n_exact_duplicates_removed"]}')
print(f'  Total após limpeza           : {dup_report["n_clean"]}')

if dup_report['conflicting_texts']:
    print()
    print('Textos com rótulos conflitantes (primeiros 5):')
    for t in dup_report['conflicting_texts'][:5]:
        print(f'  "{t[:80]}"')

## 5. Distribuição Após Limpeza

In [ ]:
clean_counts = df_clean['label'].value_counts()
clean_pct = df_clean['label'].value_counts(normalize=True) * 100

print('Distribuição após deduplicação:')
for label in ['ham', 'spam']:
    print(f'  {label:4s}: {clean_counts[label]:5d}  ({clean_pct[label]:.1f}%)')

## 6. Exemplos de Mensagens

In [ ]:
print('=== Exemplos HAM (5 aleatórios) ===')
for i, (_, row) in enumerate(df_clean[df_clean['label'] == 'ham'].sample(5, random_state=42).iterrows(), 1):
    print(f'  [{i}] {row["text"][:120]}')

print()
print('=== Exemplos SPAM (5 aleatórios) ===')
for i, (_, row) in enumerate(df_clean[df_clean['label'] == 'spam'].sample(5, random_state=42).iterrows(), 1):
    print(f'  [{i}] {row["text"][:120]}')

## 7. Análise de Vocabulário

In [ ]:
from collections import Counter

ham_words  = Counter()
spam_words = Counter()

for _, row in df_clean.iterrows():
    words = row['text'].lower().split()
    if row['label'] == 'ham':
        ham_words.update(words)
    else:
        spam_words.update(words)

print(f'Tamanho do vocabulário HAM : {len(ham_words):,}')
print(f'Tamanho do vocabulário SPAM: {len(spam_words):,}')
print(f'Total de tokens únicos     : {len(set(ham_words) | set(spam_words)):,}')

print()
print('Top-15 palavras em SPAM:')
for w, c in spam_words.most_common(15):
    print(f'  {w:20s}  {c}')

In [ ]:
# Palavras com maior razão spam/ham (mínimo 5 ocorrências em spam)
n_ham  = sum(ham_words.values())
n_spam = sum(spam_words.values())

ratios = []
for w in set(ham_words) | set(spam_words):
    freq_ham  = ham_words.get(w, 0) / n_ham
    freq_spam = spam_words.get(w, 0) / n_spam
    if freq_ham > 0 and freq_spam > 0 and spam_words.get(w, 0) >= 5:
        ratios.append((w, freq_spam / freq_ham, spam_words[w]))

ratios.sort(key=lambda x: -x[1])
print('Top-20 palavras mais associadas a SPAM (razão spam/ham):')
for w, r, count in ratios[:20]:
    print(f'  {w:25s}  razão={r:6.1f}x  freq_spam={count}')

## 8. Resumo Final

In [ ]:
print('=== Resumo do Dataset (após deduplicação) ===')
print(f'Total de mensagens: {len(df_clean):,}')
print()
for label in ['ham', 'spam']:
    sub = df_clean[df_clean['label'] == label]
    wc  = sub['text'].str.split().str.len()
    print(f'{label.upper()}')
    print(f'  n          = {len(sub):,}')
    print(f'  palavras   — média: {wc.mean():.1f}, mediana: {wc.median():.0f}, max: {wc.max()}')
    print()

print('Tamanho max. de sequência configurado: 100 tokens')
wc_all = df_clean['text'].str.split().str.len()
print(f'Cobertura com max_length=100: {(wc_all <= 100).mean()*100:.1f}% das mensagens')